In [2]:
import pandas as pd
df = pd.read_csv('selloutdata.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()


Shape: (863349, 18)

Columns: ['תאריך', 'רשת', 'סניף', 'קוד_סניף', 'קוד_פריט', 'תיאור_פריט', 'מותג', 'קבוצה', 'סוג_פריט', 'סטטוס_פריט', 'כמות_מכירות', 'ערך_מכירות', 'כמות_מלאי', 'ערך_מלאי', 'מלאי_תקף_ליום', 'כמות_כניסות', 'כמות_החזרות', 'מחיר_עלות']


C:\Users\betty\AppData\Local\Temp\ipykernel_20416\3704823554.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('selloutdata.csv')


,תאריך,רשת,סניף,קוד_סניף,קוד_פריט,תיאור_פריט,מותג,קבוצה,סוג_פריט,סטטוס_פריט,כמות_מכירות,ערך_מכירות,כמות_מלאי,ערך_מלאי,מלאי_תקף_ליום,כמות_כניסות,כמות_החזרות,מחיר_עלות
0,2026-07-18,רשת A,סניף A-38,NaN,6932554432454,מצלמת אבטחה 4K HDR אלחוטית 360° Xiaomi Smart C...,NaN,NaN,NaN,NaN,0,0.00,2,310.86,False,NaN,NaN,NaN
1,2026-07-18,רשת A,סניף A-38,NaN,6941948702028,רמקול Xiaomi Sound Outdoor Speaker 30W Black,NaN,NaN,NaN,NaN,0,0.00,3,481.49,False,NaN,NaN,NaN
2,2026-07-18,רשת A,סניף A-38,NaN,6941812751985,Xiaomi Body Composition Scale S400,NaN,NaN,NaN,NaN,1,70.32,3,210.95,False,NaN,NaN,NaN
3,2026-07-18,רשת A,סניף A-38,NaN,7290019526803,"קופסת מתנות שיאומי (מכילה בקבוק, אוזניות, מחז...",NaN,NaN,NaN,NaN,0,0.00,1,0.00,False,NaN,NaN,NaN
4,2026-07-18,רשת A,סניף A-38,NaN,6941812777565,אוזניות אלחוטיות Redmi Buds 6 Active - ורוד שקוף,NaN,NaN,NaN,NaN,0,0.00,1,59.22,False,NaN,NaN,NaN


In [3]:
df = pd.read_csv('selloutdata.csv', dtype={'קוד_פריט': str})
df['תאריך'] = pd.to_datetime(df['תאריך'], errors='coerce')
num_cols = ['כמות_מכירות', 'ערך_מכירות', 'כמות_מלאי', 'ערך_מלאי']
df[num_cols] = df[num_cols].fillna(0)
print(f"טווח תאריכים: מ-{df['תאריך'].min().date()} עד {df['תאריך'].max().date()}")
print(f"מספר רשתות: {df['רשת'].nunique()}")
print(f"מספר סניפים: {df['סניף'].nunique()}")
print(f"מספר מוצרים שונים: {df['קוד_פריט'].nunique()}")

טווח תאריכים: מ-2026-07-18 עד 2026-09-01
מספר רשתות: 9
מספר סניפים: 471
מספר מוצרים שונים: 3224


In [4]:
top_products = df.groupby(['קוד_פריט', 'תיאור_פריט']).agg(
    סך_יחידות_שנמכרו=('כמות_מכירות', 'sum'),
    סך_הכנסות=('ערך_מכירות', 'sum')
).reset_index()

top_10 = top_products.sort_values(by='סך_יחידות_שנמכרו', ascending=False).head(10)

top_10

,קוד_פריט,תיאור_פריט,סך_יחידות_שנמכרו,סך_הכנסות
1644,371559,מטען קיר שחור Anker Anker Nano Charger (45W),645,56169.44
1722,375671,אוזניות אלחוטיות Anker soundcore R60i NC Black,635,100488.04
2161,5683225,שוטף שואב אריגים BISSELL Spot Clean HydroSteam...,634,332772.99
954,328557,כבל 1 מטר Anker 322 USB-C to USB-C Black,603,20674.32
610,263073,סבון 1 ליטר לשוטף אריגים BISSELL 1084N,530,25013.29
1098,344590,אוזניות אלחוטיות Anker soundcore K20i Blue,515,44283.63
953,328556,כבל 1 מטר Anker 322 USB-C to USB-C White,420,14398.83
1297,354775,Xiaomi TV Box S (3rd Gen) EU,408,60581.96
1724,375672,אוזניות אלחוטיות Anker soundcore R60i NC White,388,61592.34
609,263073,נוזל לניקוי כתמים למכונות ניקוי BISSELL 1084N,343,17520.54


In [13]:
# 1. חישוב סך המכירות וקצב יומי לכל סניף ומוצר
total_days = (df['תאריך'].max() - df['תאריך'].min()).days + 1

sales_summary = df.groupby(['רשת', 'סניף', 'קוד_פריט', 'תיאור_פריט']).agg(
    סך_מכירות=('כמות_מכירות', 'sum')
).reset_index()

sales_summary['קצב_יומי'] = sales_summary['סך_מכירות'] / total_days

# 2. שליפת המלאי ביום האחרון (מלאי נוכחי)
last_date = df['תאריך'].max()
current_stock = df[df['תאריך'] == last_date][['סניף', 'קוד_פריט', 'כמות_מלאי']].rename(
    columns={'כמות_מלאי': 'מלאי_נוכחי'}
)

# 3. איחוד המכירות והמלאי
inventory_status = pd.merge(
    sales_summary, 
    current_stock, 
    on=['סניף', 'קוד_פריט'], 
    how='left'
).fillna({'מלאי_נוכחי': 0})

# 4. חישוב ימי המלאי (Days of Supply)
def calc_days_of_supply(row):
    stock = row['מלאי_נוכחי']
    velocity = row['קצב_יומי']
    
    if velocity > 0:
        return round(stock / velocity, 1)
    elif stock > 0:
        return 999.0  # יש מלאי אבל 0 מכירות (מלאי שתקוע ל"נצח")
    else:
        return 0.0

inventory_status['ימי_מלאי'] = inventory_status.apply(calc_days_of_supply, axis=1)

# נציג דוגמה של 5 שורות לראות שהכל הגיוני
inventory_status.head()

,רשת,סניף,קוד_פריט,תיאור_פריט,סך_מכירות,קצב_יומי,מלאי_נוכחי,ימי_מלאי
0,רשת A,סניף A-01,194644034962,אוזניות קשת אלחוטיות Anker Soundcore Space One...,0,0.000000,1.0,999.0
1,רשת A,סניף A-01,194644037345,אוזניות קשת אלחוטיות Anker Soundcore Space One...,0,0.000000,1.0,999.0
2,רשת A,סניף A-01,194644040437,עמדת טעינה עוצמתית 6 יציאות Anker Prime Charge...,0,0.000000,1.0,999.0
3,רשת A,סניף A-01,194644041007,מטען קיר Anker Prime Charger (200W) 6 Ports,1,0.021739,0.0,0.0
4,רשת A,סניף A-01,194644139810,אוזניות קשת אלחוטיות Anker Soundcore Space One...,0,0.000000,1.0,999.0


In [14]:
# 1. סינון סניפים מקבלים (ביקוש גבוה, מלאי אוזל)
receivers = inventory_status[
    (inventory_status['סך_מכירות'] >= 3) & 
    (inventory_status['ימי_מלאי'] <= 3)
][['רשת', 'קוד_פריט', 'תיאור_פריט', 'סניף', 'סך_מכירות', 'קצב_יומי', 'מלאי_נוכחי']].rename(
    columns={
        'סניף': 'סניף_מקבל',
        'סך_מכירות': 'מכירות_מקבל',
        'מלאי_נוכחי': 'מלאי_מקבל'
    }
)

# 2. סינון סניפים תורמים (מלאי קיים, ביקוש נמוך/אפסי)
donors = inventory_status[
    (inventory_status['מלאי_נוכחי'] >= 3) & 
    ((inventory_status['ימי_מלאי'] >= 40) | (inventory_status['סך_מכירות'] <= 1))
][['רשת', 'קוד_פריט', 'סניף', 'סך_מכירות', 'מלאי_נוכחי']].rename(
    columns={
        'סניף': 'סניף_תורם',
        'סך_מכירות': 'מכירות_תורם',
        'מלאי_נוכחי': 'מלאי_תורם'
    }
)

# 3. הצלבה: חיבור בין תורם למקבל באותה רשת ועבור אותו מוצר
matches = pd.merge(receivers, donors, on=['רשת', 'קוד_פריט'], how='inner')

# מוודאים שסניף לא מעביר סחורה לעצמו
matches = matches[matches['סניף_מקבל'] != matches['סניף_תורם']]

print(f"נמצאו {len(matches)} התאמות אפשריות של תורם-מקבל!")
matches[['רשת', 'תיאור_פריט', 'סניף_תורם', 'מלאי_תורם', 'סניף_מקבל', 'מלאי_מקבל']].head(5)

נמצאו 3883 התאמות אפשריות של תורם-מקבל!


,רשת,תיאור_פריט,סניף_תורם,מלאי_תורם,סניף_מקבל,מלאי_מקבל
0,רשת A,מטען קיר Anker Anker Nano Charger (45W)שחור,סניף A-05,3.0,סניף A-04,0.0
1,רשת A,מטען קיר Anker Anker Nano Charger (45W)שחור,סניף A-11,3.0,סניף A-04,0.0
2,רשת A,מטען קיר Anker Anker Nano Charger (45W)שחור,סניף A-21,3.0,סניף A-04,0.0
3,רשת A,מטען קיר Anker Anker Nano Charger (45W)שחור,סניף A-05,3.0,סניף A-38,0.0
4,רשת A,מטען קיר Anker Anker Nano Charger (45W)שחור,סניף A-11,3.0,סניף A-38,0.0


In [15]:
# מיון לפי הסניפים המקבלים הכי חזקים (שמכרו הכי הרבה)
top_clear_examples = matches.sort_values(by='מכירות_מקבל', ascending=False)

# נציג 10 דוגמאות מובהקות
top_clear_examples[[
    'רשת', 
    'תיאור_פריט', 
    'סניף_מקבל', 
    'מכירות_מקבל', 
    'מלאי_מקבל', 
    'סניף_תורם', 
    'מלאי_תורם', 
    'מכירות_תורם'
]].drop_duplicates(subset=['רשת', 'תיאור_פריט', 'סניף_מקבל']).head(10)

,רשת,תיאור_פריט,סניף_מקבל,מכירות_מקבל,מלאי_מקבל,סניף_תורם,מלאי_תורם,מכירות_תורם
3765,רשת H,שוטף שואב אריגים BISSELL Spot Clean HydroSteam...,סניף H-04,506,-6.0,סניף H-07,33.0,0
3757,רשת H,סבון 1 ליטר לשוטף אריגים BISSELL 1084N,סניף H-04,428,-4.0,סניף H-03,3.0,0
3764,רשת H,שוטף שואב אריגים BISSELL Spot Clean HydroSteam...,סניף H-02,128,1.0,סניף H-07,33.0,0
3755,רשת H,סבון 1 ליטר לשוטף אריגים BISSELL 1084N,סניף H-02,102,0.0,סניף H-03,3.0,0
1549,רשת C,שעון Xiaomi Smart Band 9 Active Black,סניף C-77,74,0.0,סניף C-13,10.0,0
3456,רשת E,כיריים MIDEA גז 4 להבות אמייל 60SJ401 6740 ניר...,סניף E-14,66,0.0,סניף E-23,3.0,-1
2603,רשת C,מאוורר מגדל 12 מהירויות מנוע DC 24W MIDEA FZ10...,סניף C-04,49,0.0,סניף C-74,3.0,0
817,רשת C,"מאוורר מגדל 42"" MIDEA FZ10-18TRB",סניף C-04,40,0.0,סניף C-07,3.0,0
582,רשת C,מיקרוגל 29דיגיטלי ליטר עם גריל MIDEA EG9P032MX...,סניף C-04,34,0.0,סניף C-03,3.0,0
1444,רשת C,Redmi Watch 5 Active Silver שעון חכם,סניף C-04,32,0.0,סניף C-96,3.0,0


In [18]:
# 1. חישוב קצב מכירות יומי אמיתי לכל סניף מקבל (סך מכירות חלקי 45 יום)
rec_true_velocity = df.groupby(['רשת', 'סניף', 'קוד_פריט', 'תיאור_פריט']).agg(
    סך_מכירות_בפועל=('כמות_מכירות', 'sum'),
    ימי_חוסר=('כמות_מלאי', lambda x: (x <= 0).sum())  # ספירת כל הימים שהמלאי היה 0 או שלילי
).reset_index().rename(columns={'סניף': 'סניף_מקבל'})

# קצב יומי על פני כל 45 הימים
rec_true_velocity['קצב_יומי_אמיתי'] = rec_true_velocity['סך_מכירות_בפועל'] / 45.0

# 2. סינון רק לסניפים המקבלים שהוגדרו בטבלת הניודים (matches)
unique_receivers = matches[['רשת', 'סניף_מקבל', 'קוד_פריט']].drop_duplicates()
loss_fixed = pd.merge(unique_receivers, rec_true_velocity, on=['רשת', 'סניף_מקבל', 'קוד_פריט'], how='inner')

# 3. חיבור מחיר יחידה
loss_fixed = pd.merge(loss_fixed, item_prices[['קוד_פריט', 'מחיר_יחידה']], on='קוד_פריט', how='left')

# 4. חישוב אובדן מכירות אמיתי
# יחידות שאבדו = ימי חוסר * קצב יומי
loss_fixed['יחידות_שאבדו'] = (loss_fixed['ימי_חוסר'] * loss_fixed['קצב_יומי_אמיתי']).round(1)
loss_fixed['הפסד_מצטבר_אמיתי_שח'] = (loss_fixed['יחידות_שאבדו'] * loss_fixed['מחיר_יחידה']).round(2)
loss_fixed['הפסד_יומי_שוטף_שח'] = (loss_fixed['קצב_יומי_אמיתי'] * loss_fixed['מחיר_יחידה']).round(2)

# סינון שורות ללא אובדן
loss_fixed = loss_fixed[loss_fixed['הפסד_מצטבר_אמיתי_שח'] > 0].sort_values(by='הפסד_מצטבר_אמיתי_שח', ascending=False)

# תוצאות
real_total_lost = loss_fixed['הפסד_מצטבר_אמיתי_שח'].sum()
real_daily_lost = loss_fixed['הפסד_יומי_שוטף_שח'].sum()

print("=" * 60)
print(f"סה\"כ הפסד מכירות מתוקן ב-45 הימים: ₪{real_total_lost:,.2f}")
print(f"קצב הפסד יומי שוטף: ₪{real_daily_lost:,.2f}")
print("=" * 60)

loss_fixed[[
    'רשת', 'סניף_מקבל', 'תיאור_פריט', 'סך_מכירות_בפועל', 
    'ימי_חוסר', 'קצב_יומי_אמיתי', 'הפסד_יומי_שוטף_שח', 'הפסד_מצטבר_אמיתי_שח'
]].head(10).style.format({
    'קצב_יומי_אמיתי': '{:.2f}',
    'הפסד_יומי_שוטף_שח': '₪{:,.2f}',
    'הפסד_מצטבר_אמיתי_שח': '₪{:,.2f}'
})

סה"כ הפסד מכירות מתוקן ב-45 הימים: ₪401,018.89
קצב הפסד יומי שוטף: ₪21,481.93


,רשת,סניף_מקבל,תיאור_פריט,סך_מכירות_בפועל,ימי_חוסר,קצב_יומי_אמיתי,הפסד_יומי_שוטף_שח,הפסד_מצטבר_אמיתי_שח
394,רשת H,סניף H-04,שוטף שואב אריגים BISSELL Spot Clean HydroSteam SEL,506,41,11.24,"₪5,901.97","₪241,969.00"
393,רשת H,סניף H-02,שוטף שואב אריגים BISSELL Spot Clean HydroSteam SEL,128,33,2.84,"₪1,492.99","₪49,286.09"
387,רשת H,סניף H-04,סבון 1 ליטר לשוטף אריגים BISSELL 1084N,428,41,9.51,₪463.40,"₪19,001.37"
79,רשת C,סניף C-04,תנור בנוי MIDEA 65DME30007S/6770 נירוסטה,31,19,0.69,₪356.42,"₪6,777.66"
254,רשת C,סניף C-04,מאוורר מגדל 12 מהירויות מנוע DC 24W MIDEA FZ10-25ARD,49,27,1.09,₪186.18,"₪5,026.76"
389,רשת H,סניף H-04,שואב שוטף BISSELL 4006N Crosswave Wave OmniFind אל,28,13,0.62,₪352.92,"₪4,594.22"
124,רשת C,סניף C-04,שואב אבק נטען Xiaomi Vacuum Cleaner G20,18,16,0.40,₪214.02,"₪3,424.39"
183,רשת C,סניף C-04,שואב רובוטי Anker Eufy E20 Omni,14,13,0.31,₪241.67,"₪3,107.16"
66,רשת C,סניף C-04,מיקרוגל 29דיגיטלי ליטר עם גריל MIDEA EG9P032MX אפור כהה,34,13,0.76,₪234.32,"₪3,039.26"
98,רשת C,סניף C-04,"מאוורר מגדל 42"" MIDEA FZ10-18TRB",40,21,0.89,₪131.01,"₪2,756.14"
